# ADEPT SDK - Getting Started

This notebook demonstrates the ADEPT SDK for building applications on the framework.

## Security Architecture

All SDK operations route through `nginx_proxy` (TLS + JWT authentication).
JupyterLab is isolated on `frontend_net` — the SDK connects via the proxy.

## SDK Components
- Client initialization and authentication
- Synchronous and asynchronous operations
- Response handling and error management
- Session management
- Tool registration

## Docker Configuration
- JupyterLab service: `deployment/local/docker-compose/docker-compose.core.yaml`
- Rebuild after config changes: `make rebuild-jupyterlab`

In [ ]:
# Install SDK if needed
import sys
!{sys.executable} -m pip install -q /app/src/agentic_framework_sdk/

import os
import json
import httpx
from agentic_framework_sdk import OpenAI
from agentic_framework_sdk.auth import KeycloakAuth

# Load config from setup notebooks
config = json.load(open("/tmp/adept_config.json"))
NGINX_URL = config.get('nginx_url', 'https://nginx_proxy')
API_BASE_URL = config.get('api_base_url', f'{NGINX_URL}/v1')
KEYCLOAK_TOKEN_URL = config.get('keycloak_token_url', f'{NGINX_URL}/realms/master/protocol/openid-connect/token')

print(f"SDK configured for: {API_BASE_URL}")
print(f"Auth via: {NGINX_URL}/realms/master/...")

In [ ]:
# Initialize SDK client — routes through nginx_proxy
auth = KeycloakAuth(
    keycloak_url=NGINX_URL,
    realm="master",
    client_id="login-client",
    verify_ssl=False
)

# Authenticate
token = auth.get_token(
    username="notebook-user",
    password=os.getenv("NOTEBOOK_USER_PASSWORD", "notebook-password")
)

# Create client
client = OpenAI(
    base_url=API_BASE_URL,
    api_key=token,
    verify_ssl=False
)

print(f"SDK Client initialized")
print(f"   Version: {client.version}")
print(f"   Routing: nginx_proxy (TLS + JWT)")

## Using the SDK

In [ ]:
# Send a chat message
response = sdk_client.chat.completions.create(
    model="agentic-framework:scientific-agent-v1",
    messages=[
        {"role": "user", "content": "What tools do you have available?"}
    ]
)

print(response.choices[0].message.content)